# Análise de Índices de Vegetação

Pipeline de visão computacional para cálculo de VARI e ExG a partir de imagens RGB.

## Etapa 1 — Carregar imagem e converter BGR → RGB

In [1]:
import cv2
import numpy as np

# cv2.imread retorna um array NumPy (H, W, 3) em BGR com dtype uint8
img_bgr = cv2.imread('../images/paisagem-verde.jpeg')

# OpenCV carrega em BGR; Matplotlib exibe em RGB — a conversão corrige a ordem dos canais
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

## Etapa 2 — Inspecionar shape, dtype e pixel

In [2]:
# shape retorna (altura, largura, canais) — eixo 0 é vertical, eixo 1 horizontal
print('Shape:', img_rgb.shape)

# dtype indica o tipo de cada elemento; uint8 = inteiro sem sinal de 0 a 255
print('Dtype:', img_rgb.dtype)

# img[y, x] acessa o pixel na linha y, coluna x — retorna [R, G, B]
pixel_y, pixel_x = 100, 200
print(f'Pixel [{pixel_y}, {pixel_x}]:', img_rgb[pixel_y, pixel_x])

Shape: (4032, 2268, 3)
Dtype: uint8
Pixel [100, 200]: [20 34 19]


## Etapa 3 — Separar canais R, G, B

In [ ]:
# Converte para float32 antes de separar — evita overflow em uint8 nos cálculos de índice
img_f = img_rgb.astype(np.float32)

# Fatia o eixo 2: [:, :, n] = todas as linhas, todas as colunas, canal n
R = img_f[:, :, 0]  # vermelho
G = img_f[:, :, 1]  # verde
B = img_f[:, :, 2]  # azul

print('Shape de cada canal:', R.shape)
print(f'R — min: {R.min():.0f}, max: {R.max():.0f}')
print(f'G — min: {G.min():.0f}, max: {G.max():.0f}')
print(f'B — min: {B.min():.0f}, max: {B.max():.0f}')

## Etapa 4 — Calcular VARI

In [ ]:
# Numerador: pixels com G > R são positivos (vegetação); G < R são negativos (solo, estruturas)
# Denominador: 1e-6 evita divisão por zero quando G + R - B = 0
vari = (G - R) / (G + R - B + 1e-6)

# Limita ao intervalo [-1, 1] — outliers de pixels saturados não distorcem o colormap
vari = np.clip(vari, -1, 1)

print('VARI shape:', vari.shape)
print(f'VARI — min: {vari.min():.4f}, max: {vari.max():.4f}, média: {vari.mean():.4f}')

## Etapa 5 — Calcular ExG

In [ ]:
# ExG não é normalizado — valores dependem da escala 0-255, range teórico: -510 a +510
# Valores positivos = verde dominante (vegetação); negativos = vermelho/azul dominante
exg = 2 * G - R - B

print('ExG shape:', exg.shape)
print(f'ExG — min: {exg.min():.1f}, max: {exg.max():.1f}, média: {exg.mean():.2f}')
print(f'\nComparação de médias:')
print(f'  VARI (normalizado -1 a 1): {vari.mean():.4f}')
print(f'  ExG  (escala 0-255):       {exg.mean():.2f}')

## Etapa 6 — Visualização comparativa

In [ ]:
import matplotlib.pyplot as plt

# 1 linha, 3 colunas — largura 18" comporta três mapas de imagem 4K sem comprimir
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Imagem original ---
axes[0].imshow(img_rgb)
axes[0].set_title('Imagem original (RGB)', fontsize=13)
axes[0].axis('off')

# --- Mapa VARI ---
# vmin/vmax ancoram a escala ao intervalo teórico do índice, não ao min/max desta imagem
im_vari = axes[1].imshow(vari, cmap='RdYlGn', vmin=-1, vmax=1)
axes[1].set_title('VARI  (Visible Atmospherically Resistant Index)', fontsize=13)
axes[1].axis('off')
plt.colorbar(im_vari, ax=axes[1], fraction=0.046, pad=0.04)

# --- Mapa ExG ---
# ExG não é normalizado — deixa Matplotlib escalar pelo min/max real da imagem
im_exg = axes[2].imshow(exg, cmap='RdYlGn')
axes[2].set_title('ExG  (Excess Green Index)', fontsize=13)
axes[2].axis('off')
plt.colorbar(im_exg, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()

# Salva resultado em outputs/ com resolução adequada para apresentação
plt.savefig('../outputs/comparativo_indices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/comparativo_indices.png')